# Replication Code 

This work is a replication of the paper "Using Large Language Models to Simulate Multiple Humans and Replicate Human Subject Studies" by Gati Aher, Rosa I. Arriaga, and Adam Tauman Kalai (2023). In the original paper, the authors demonstrated results on four well-known psychology experiments using large language models (LLMs) to simulate human participants:

1. The Ultimatum Game - a behavioral economics game evaluating fairness and rationality. The LLM simulated responders deciding to accept or reject offers.

2. Garden Path Sentences - a psycholinguistics experiment judging the grammaticality of ambiguous sentences. The LLM simulated participants' judgments. 

3. Milgram Shock Experiment - a famous social psychology obedience study. The LLM simulated administering shocks.

4. Wisdom of Crowds - an experiment evaluating collective intelligence and wisdom. The LLM simulated participants' numerical estimates.

The code replication will focus on designing each experiment simulation in Python. The high-level steps are:

1. Define the experimental conditions and stimuli as input variables. For example, offer amounts for the Ultimatum Game.

2. Write prompt templates that incorporate the inputs. Prompts should provide clear instructions and examples. 

3. Query the LLM using the prompts. The API call returns generated text.

4. Parse the generated text to record outcomes of interest. For example, extracting accept/reject decisions.


The replication will apply this methodology to simulate each experiment with the GPT-3 API. While API costs limit large-scale runs, the code will demonstrate the approach and allow small-scale experiments to be reproduced. The prompts and outcome parsing will be designed to match the original paper as closely as possible.

[https://arxiv.org/abs/2208.10264](Arxiv Link)


# Imports

In [2]:
import openai
import os
import random

openai.api_key = ""


## Data Prep
The original surnames data taken from the article

In [4]:
surnames_dict = {
    "American Indian": [
        "Begay", "Yazzie", "Benally", "Tsosie", "Nez", "Begaye", "Etsitty", "Becenti", 
        "Yellowhair", "Manygoats", "Wauneka", "Manuelito", "Apachito", "Bedonie", "Calabaza", 
        "Peshlakai", "Claw", "Roanhorse", "Goldtooth", "Etcitty", "Tsinnijinnie", "Notah", 
        "Clah", "Atcitty", "Twobulls", "Werito", "Hosteen", "Yellowman", "Attakai", "Bitsui", 
        "Delgarito", "Henio", "Goseyun", "Keams", "Secatero", "Declay", "Tapaha", "Beyale", 
        "Haskie", "Cayaditto", "Blackhorse", "Ethelbah", "Tsinnie", "Walkingeagle", "Altaha", 
        "Bitsilly", "Wassillie", "Benallie", "Smallcanyon", "Littledog", "Cosay", "Clitso", 
        "Tessay", "Secody", "Bigcrow", "Tabaha", "Chasinghawk", "Blueeyes", "Olanna", "Blackgoat", 
        "Cowboy", "Kanuho", "Shije", "Gishie", "Littlelight", "Laughing", "Whitehat", "Eriacho", 
        "Runningcrane", "Chinana", "Kameroff", "Spottedhorse", "Arcoren", "Whiteplume", "Dayzie", 
        "Spottedeagle", "Heavyrunner", "Standingrock", "Poorbear", "Ganadonegro", "Ayze", 
        "Whiteface", "Yepa", "Talayumptewa", "Madplume", "Bitsuie", "Tsethlikai", "Ahasteen", 
        "Dosela", "Birdinground", "Todacheenie", "Bitsie", "Todacheene", "Bullbear", "Lasiloo", 
        "Keyonnie", "Notafraid", "Colelay", "Kallestewa", "Littlewhiteman"
    ],
    "ANHOPI": [
        "Nguyen", "Kim", "Patel", "Tran", "Chen", "Li", "Le", "Wang", "Yang", "Pham", "Lin", "Liu", 
        "Huang", "Wu", "Zhang", "Shah", "Huynh", "Yu", "Choi", "Ho", "Kaur", "Vang", "Chung", "Truong", 
        "Phan", "Xiong", "Lim", "Vo", "Vu", "Lu", "Tang", "Cho", "Ngo", "Cheng", "Kang", "Tan", "Ng", 
        "Dang", "Do", "Ly", "Han", "Hoang", "Bui", "Sharma", "Chu", "Ma", "Xu", "Zheng", "Song", "Duong", 
        "Liang", "Sun", "Zhou", "Thao", "Zhao", "Shin", "Zhu", "Leung", "Hu", "Jiang", "Lai", "Gupta", 
        "Cheung", "Desai", "Oh", "Ha", "Cao", "Yi", "Hwang", "Lo", "Dinh", "Hsu", "Chau", "Yoon", "Luu", 
        "Trinh", "He", "Her", "Luong", "Mehta", "Moua", "Tam", "Ko", "Kwon", "Yoo", "Chiu", "Su", "Shen", 
        "Pan", "Dong", "Begum", "Gao", "Guo", "Chowdhury", "Vue", "Thai", "Jain", "Lor", "Yan", "Dao"
    ],
    "African American": [
        "Smalls", "Jeanbaptiste", "Diallo", "Kamara", "Pierrelouis", "Gadson", "Jeanlouis", "Bah", 
        "Desir", "Mensah", "Boykins", "Chery", "Jeanpierre", "Boateng", "Owusu", "Jama", "Jalloh", 
        "Sesay", "Ndiaye", "Abdullahi", "Wigfall", "Bienaime", "Diop", "Edouard", "Toure", "Grandberry", 
        "Fluellen", "Manigault", "Abebe", "Sow", "Traore", "Mondesir", "Okafor", "Bangura", "Louissaint", 
        "Cisse", "Osei", "Calixte", "Cephas", "Belizaire", "Fofana", "Koroma", "Conteh", "Straughter", 
        "Jeancharles", "Mwangi", "Kebede", "Mohamud", "Prioleau", "Yeboah", "Appiah", "Ajayi", "Asante", 
        "Filsaime", "Hardnett", "Hyppolite", "Saintlouis", "Jeanfrancois", "Ravenell", "Keita", "Bekele", 
        "Tadesse", "Mayweather", "Okeke", "Asare", "Ulysse", "Saintil", "Tesfaye", "Jeanjacques", "Ojo", 
        "Nwosu", "Okoro", "Fobbs", "Kidane", "Petitfrere", "Yohannes", "Warsame", "Lawal", "Desta", 
        "Veasley", "Addo", "Leaks", "Gueye", "Mekonnen", "Stfleur", "Balogun", "Adjei", "Opoku", "Coaxum", 
        "Vassell", "Prophete", "Lesane", "Metellus", "Exantus", "Hailu", "Dorvil", "Frimpong", "Berhane", 
        "Njoroge", "Beyene"
    ],
    "Latino": [
        "Garcia", "Rodriguez", "Martinez", "Hernandez", "Lopez", "Gonzalez", "Perez", "Sanchez", 
        "Ramirez", "Torres", "Flores", "Rivera", "Gomez", "Diaz", "Morales", "Gutierrez", "Ortiz", 
        "Chavez", "Ruiz", "Alvarez", "Castillo", "Jimenez", "Vasquez", "Moreno", "Herrera", "Medina", 
        "Aguilar", "Vargas", "Guzman", "Mendez", "Munoz", "Salazar", "Garza", "Soto", "Vazquez", 
        "Alvarado", "Delgado", "Pena", "Contreras", "Sandoval", "Guerrero", "Rios", "Estrada", "Ortega", 
        "Nunez", "Maldonado", "Dominguez", "Vega", "Espinoza", "Rojas", "Marquez", "Padilla", "Mejia", 
        "Juarez", "Figueroa", "Avila", "Molina", "Campos", "Ayala", "Carrillo", "Cabrera", "Lara", "Robles", 
        "Cervantes", "Solis", "Salinas", "Fuentes", "Velasquez", "Aguirre", "Ochoa", "Cardenas", "Calderon", 
        "Rivas", "Serrano", "Rosales", "Castaneda", "Gallegos", "Ibarra", "Suarez", "Orozco", "Salas", 
        "Escobar", "Velazquez", "Macias", "Zamora", "Villarreal", "Barrera", "Pineda", "Santana", "Trevino", 
        "Lozano", "Rangel", "Arias", "Mora", "Valenzuela", "Zuniga", "Melendez", "Galvan", "Velez", "Meza"
    ],
    "White": [
        "Olson", "Snyder", "Wagner", "Meyer", "Schmidt", "Ryan", "Hansen", "Hoffman", "Johnston", "Larson", 
        "Carlson", "Obrien", "Jensen", "Hanson", "Weber", "Walsh", "Schultz", "Schneider", "Keller", "Beck", 
        "Schwartz", "Becker", "Wolfe", "Zimmerman", "Mccarthy", "Erickson", "Klein", "Oconnor", "Swanson", 
        "Christensen", "Fischer", "Wolf", "Gallagher", "Schroeder", "Parsons", "Bauer", "Mueller", "Hartman", 
        "Kramer", "Flynn", "Owen", "Shaffer", "Hess", "Olsen", "Petersen", "Roth", "Hoover", "Weiss", "Decker", 
        "Yoder", "Larsen", "Sweeney", "Foley", "Hensley", "Huffman", "Cline", "Oneill", "Koch", "Brennan", 
        "Berg", "Russo", "Macdonald", "Kline", "Jacobson", "Berger", "Blankenship", "Bartlett", "Odonnell", 
        "Stein", "Stout", "Sexton", "Nielsen", "Howe", "Morse", "Knapp", "Herman", "Stark", "Hebert", 
        "Schaefer", "Reilly", "Conrad", "Donovan", "Mahoney", "Hahn", "Peck", "Boyle", "Hurley", "Mayer", 
        "Mcmahon", "Case", "Duffy", "Friedman", "Fry", "Dougherty", "Crane", "Huber", "Moyer", "Krueger", 
        "Rasmussen", "Brandt"
    ]
}


# Code

## The Ultimatum Game TE

Two players are matched and assigned the roles of proposer and responder. <br>
**Proposer:** given an amount of money and has to decide how to split it between himself and the responder. <br>
**Responder:** If the responder accepts the take-it-or-leave-it proposal, both players receive their designated shares, otherwise both players receive nothing <br>
**Aim:**  Experiments on the Ultimatum Game reveal an anomaly in economic decision making: since the responder will receive nothing if they reject, the responder’s dominant strategy to maximize monetary gain is to always accept; in practice, responders typically reject unfair proposals.

### Inputs
1. an integer offer in {0, 1, . . . , 10}
2. the name of the proposer
3. the name of the responder
   
The offer corresponds to an initial endowment fixed at $10 and eleven possible offers.

### Name Shufler

 Generates pairs of surnames with associated titles and random scores. It creates variations of titles (Mr. and Ms.) for each surname pair and associates them with random scores. 

In [ ]:
import random

def generate_pairs(surnames_dict):
    # Flatten the dictionary to get a list of all surnames
    all_surnames = []
    for names in surnames_dict.values():
        all_surnames.extend(names)
    random.shuffle(all_surnames)
    pairs = []
    for surname in all_surnames:
        # Get the racial group of the current surname
        current_group = next((group for group, names in surnames_dict.items() if surname in names), None)

        # For each racial group, select one surname for pairing
        for group, names in surnames_dict.items():
            paired_surname = random.choice(names)
            
            # Create 2x2 combinations of "Mr." and "Ms." titles and combine them with surnames
            pairs.append((f"Mr. {surname} , Mr. {paired_surname}", random.randint(0, 10)))
            pairs.append((f"Mr. {surname} , Ms. {paired_surname}", random.randint(0, 10)))
            pairs.append((f"Ms. {surname} , Mr. {paired_surname}", random.randint(0, 10)))
            pairs.append((f"Ms. {surname} , Ms. {paired_surname}", random.randint(0, 10)))
                
    # Return the first 10,000 pairs
    enumerated_pairs = list(enumerate(pairs, start=1))
    return enumerated_pairs[:10000]

# Test dictionary for demonstration


pairs = generate_pairs(surnames_dict)
pairs = pairs[:100]
pairs

Lets take a look at our pairs

In [ ]:
for pair_number, (pair_string, random_number) in pairs:
    names = pair_string.split(",") 
    first_name = names[0].strip()
    second_name = names[1].strip()
    print(f"Pair {pair_number}:")
    print("First Name:", first_name)
    print("Second Name:", second_name)
    print(f"Money that {first_name} will keep:", random_number)
    print(f"Money that {second_name} will keep:", 10 - random_number)
    print("-" * 20)

Using gpt-3.5-turbo-0613 model update to replicate the process since this update introduces the function calling capability for the model and makes it much easier to parse well structured outputs from the model

In [ ]:
import openai

def deal(first_name, second_name, money, basemoney):
    response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo-0613",
        messages=[
            {
                "role": "system",
                "content": "You are helpfull asistant that will give only one word answers that who will analyze the following scenario and consider what individuals do in this situation and will give a definitive answer as accepted or rejected."
            },
            {
                "role": "user",
                "content": f"""
                In the following scenario, {first_name} had to decide whether to accept or reject the proposal.
                Scenario: {second_name} is given {basemoney}. {second_name} will propose how to split the money between him/her self and {first_name}. Then {first_name} will decide whether to accept or reject {second_name}'s proposal. If {first_name} accepts, then {second_name} and {first_name} get the money as they agreed to split. if {first_name} rejects, then {second_name} and {first_name} both receive nothing. 
                The offer: {second_name} proposes to give {first_name} {offer} and keep {basemoney - offer} for him/her self.
                Answer: {first_name} decides to: 
                """
            },
        ],
        temperature=0,
    )
    
   
    print(f"The proposer is: {first_name}")
    print(f"The responder is: {second_name}")
    print(f"The offered amount is {offer}")

    return response 

# Assuming you have defined debug_pairs somewhere
for pair_number, (pair_string, random_number) in pairs:
    names = pair_string.split(",")
    first_name = names[0].strip()
    second_name = names[1].strip()
    basemoney = 10
    offer = 10 - random_number

    response = deal(first_name, second_names, money, basemoney)
    answer = response['choices'][0]['message']['content']
    print(f"The {first_name} would {answer} the offer.")
    print("-*-" * 20)


## Garden Path TE
A garden path sentence is a grammatical sentence that seems ungrammatical because it contains a word or phrase that can be interpreted in multiple ways. For example, a human reading “While Anna dressed the baby that was small and cute spit up on the bed” may initially believe that Anna is dressing the baby but upon re-parsing the sentence understand that Anna is dressing herself. Psycholinguists use garden path sentences to study the variability in the difficulty of comprehending relative clause constructions and other effects.

### Inputs
**Garden Path Sentences:**
- 12 Optionally Transitive (OT) garden path sentences.
- 12 Reflexive Absolute Transitive (RAT) garden path sentences.
- Total of 24 garden path sentences.
  
**Control Sentences:**
- 24 control sentences derived from the garden path sentences by adding a disambiguating comma after the verb of the subordinate clause.

**Names Set**
- A set of 1000 names

**Novel Garden Path Sentences**
- Set of 12 RAT garden path sentences authored by the researchers.
- Set of 12 OT garden path sentences authored by the researchers. 

The code down below will be used for name selection from the data set In the Garden Path Turing Experiment simulation, the authors selected names from a pool of 1000 first name-surname combinations 

In [43]:
sentences = [
    "While the man hunted, the deer that was brown and graceful ran into the woods.",
    "While the skipper sailed, the boat that was small and leaky veered off course.",
    "While the reporter photographed, the rocket that was silver and white sat on the launch pad.",
    "While the orchestra performed, the symphony that was short and simple played on the radio.",
    "While the student read, the notes that were long and boring blew off the desk.",
    "While Jack ordered, the fish that was silver and black cooked in a pot.",
    "While Susan wrote, the letter that was long and eloquent fell off the table.",
    "While the secretary typed, the memo that was clear and concise neared completion.",
    "While the farmer steered, the tractor that was big and green pulled the plough.",
    "While the lawyer studied, the contract that was old and wrinkled lay on the roll-top desk.",
    "As Henry whittled, the stick that was long and bumpy broke in half.",
    "While Rick drove, the car that was red and dusty veered into a ditch.",
    "While Jim bathed, the child that was blond and pudgy giggled with delight.",
    "While the chimps groomed, the baboons that were large and hairy sat in the grass.",
    "While Frank dried off, the car that was red and shiny sat in the driveway.",
    "While Betty woke up, the neighbor that was old and cranky coughed loudly.",
    "While the thief hid, the jewelry that was elegant and expensive sparkled brightly.",
    "While Anna dressed, the baby that was small and cute spit up on the bed.",
    "While the boy washed, the dog that was white and furry barked loudly.",
    "While the jockey settled down, the horse that was sleek and brown stood in the stall.",
    "While the mother undressed, the baby that was bald and helpless cried softly.",
    "While the nurse shaved, the patient that was tired and weak watched TV.",
    "While the girl scratched, the cat that was grey and white stared at the dog.",
    "While the mother calmed down, the children that were tired and irritable sat on the bed.",
    "While the butler answered, the door that was large and green blew shut.",
    "While Charlie cooked, the soup that was hot and delicious cooled off.",
    "While the host decorated, the room that was barren and dark filled with people.",
    "While the child played, the game that was long and boring ended abruptly.",
    "While Catherine drank, the whiskey that was cold and smooth aged in a barrel.",
    "While the father sewed, the stuffed animal that was torn and dirty smelled afoul.",
    "While the professor strummed, the guitar that was beautiful and red remained unplayed.",
    "While the general messaged, the troops that were rested and strong approached the target.",
    "While the pilot flew, the plane that was big and white sat on the runway.",
    "While the thief stole, the laptop that was hot and running caught on fire.",
    "While the choir sang, the melody that was beautiful and serene echoed through the halls.",
    "While the lecturer taught, the students who were bored and hungry left the class.",
    "While the scientists starved, the rats that were small and white ate the cheese.",
    "While the investor exercised, the options that were old and unvested sat on the table.",
    "While the hunter laid down, the gun that was loaded and dangerous leaned against the chair.",
    "While the caretaker showered, the resident that was old and wrinkled snuck out the back.",
    "While Leo wound down, the party that was fun and silly started to get busy.",
    "While the students turned in, the homework that was long and important remained unfinished.",
    "While the picknicker stretched out, the blanket that was long and clean laid on the grass.",
    "While the teacher relaxed, the students that were loud and obnoxious made snowballs.",
    "While the cheerleaders cheered up, the crowd that was disappointed and tired abandoned their seats.",
    "While the cook soaked, the mushrooms that were white and soft sat on the counter.",
    "While the doctor isolated, the patient that was big and impatient left the hospital.",
    "While the accountant prepared, the calculations that were important and classified leaked to the public."
]


In [ ]:
import random

def generate_pairs(surnames_list, sentences_list):
    pairs = []
    
    for surname in surnames_list:
        for sentence in sentences_list:
            pairs.append((f"Mr. {surname}", sentence))
            pairs.append((f"Ms. {surname}", sentence))
    
    return pairs

# Assuming you have a defined dictionary surnames_dict
all_surnames = [surname for names in surnames_dict.values() for surname in names][:500]

# Generate pairs
pairs = generate_pairs(all_surnames, sentences)

# Display a random sample of 50 pairs for demonstration
sample_pairs = random.sample(pairs, 100)
for pair in sample_pairs:
    print(pair[0], pair[1])


In [ ]:
for pair in sample_pairs:
    print(f'{pair[0]} will evaluate -> {pair[1]}')


In [ ]:
def gardenpath(name, sentence):
    response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo-0613",
        messages=[
            {
                "role": "system",
                "content": "You are helpfull asistant that will give only one word answers that who will analyze the following scenario and consider what individuals do in this situation and will give a definitive answer as grammatical  or ungrammatical."
            },
            {
                "role": "user",
                "content": f""" 
                {name} was asked to indicate whether the following sentence was grammatical or ungrammatical. 
                Sentence: {sentence}
                Answer: {name} indicated that the sentence was 
                """
            },
        ],
        temperature=0,
    )
    return response

for pair in sample_pairs:
    name = pair[0]
    sentence = pair[1]
    response = gardenpath(name, sentence)
    answer = response['choices'][0]['message']['content']
    print(print(f"Participant:{name} \n Sentence: {sentence} \n Result: {answer}"))
    print("-*-" * 10)

## Milgram Shock TE
The obedience to authority studies, developed by are a series of famous social experiments that aimed to find when and how people would defy authority in the face of a clear moral imperative Simulating the Milgram experiment involves a series of both free-response prompts and 2-choice prompts on each of the 30 shock levels, unless the experiment is terminated earlier. The record is built up sequentially, starting with a passage describing the information available to the subject following Milgram’s (1963) procedure.

### Inputs
The input to the multi-stage Milgram Shock simulator is a subject’s name. To get a diverse and balanced pool of subjects, we took the top 10 most common surnames from each racial group and both Mr. and Ms. titles, yielding 100 uniquely named subjects.

### Simulator
Simulator. This initialization text describes the setup of the experiment with pertinent details to convince the subject of the experiment’s legitimacy, instructions on when to shock and not shock the victim, and a predetermined preliminary run. Then, the record is elongated by appending an interleaving series of: (a) pre-specified narrations of the learners actions, (b) text describing the subject’s behavior (generated using the LM), and (c) canned phrases said by the experimenter when the subject exhibits disobedient or questioning behavior. 


Name selector for Miligram TE

In [ ]:
import random

def generate_pairs_with_voltages(surnames_list):
    pairs_with_voltages = []
    
    for surname in surnames_list:
        for voltage in range(15, 301, 15):
            pairs_with_voltages.append((f"Mr. {surname}", voltage))
            pairs_with_voltages.append((f"Ms. {surname}", voltage))
    
    return pairs_with_voltages

# Assuming you have a defined dictionary surnames_dict where keys are racial groups and values are lists of surnames

pairs_with_voltages = []

# Generate pairs with voltages from each racial group
for racial_group, surnames_list in surnames_dict.items():
    selected_surnames = random.sample(surnames_list, 10)  # Pick 10 surnames randomly from each group
    pairs_with_voltages.extend(generate_pairs_with_voltages(selected_surnames))

pairs_with_voltages = pairs_with_voltages[:80]# Print generated pairs with voltages
for name, voltage in pairs_with_voltages:
    print(f"{name} - {voltage} volts")


In [ ]:
def MilEx(name, voltage):
    response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo-0613",
        messages=[
            {
                "role": "system",
                "content": "You are an observer in the classical miligram experiment. You are asked to indicate whether the participant will continue to administer shocks to the learner or not Do not indicate that you are an AI model based on real world human data make a prediction based on the information provided. Make sure that you are analyzing the situation from the perspective of the participant in a real world situation. At the end of the analysis, simply only indicate whether the participant will continue to administer shocks to the learner or not. "
            },
            {
                "role": "user",
                "content": f""" 
                The current punishment voltage level is {voltage}. The experimenter is sitting next to {name} The learner selects a wrong answer If the participant decides not to continue, the experiment will end. If the participant decides to continue, the voltage level will be increased by 15 volts. If the participant reject the administor the shock the experimenter will push participant saying things like: it is absolutely essential that you continue. Any voltage level above 200 is considered leathal do not forget that participant can quit anytime 
                {name} is asked to continue. 
                Now put yourself in the shoes of {name} and decide whether {name} will continue or not. Simply indicate whether {name} will continue or not. nothing else.
                {name} decides: continue or not continue?
                """
            },
        ],
        temperature=0,
    )
    return response

for name, voltage in pairs_with_voltages:
    name = name
    voltage = voltage
    response = MilEx(name, voltage)
    answer = response['choices'][0]['message']['content']
    print((f"Participant:{name} \n Voltage level: {voltage} \n Result: {answer}"))
    print("-*-" * 10)

The findings of the experiment above is interesting and worth investigating further

## Wisdom of Crowds TE
Aggregating responses from a diverse crowd often yields very accurate estimates of quantities, even when individual estimates have high error.
What they wanted to measure: How accurately the language models could simulate this wisdom of crowds phenomenon when estimating general knowledge questions.

In [ ]:
def generate_name_combinations(surnames_dict):
    titles = ['Mr.', 'Ms.', 'Mx.']
    all_surnames = [surname for names in surnames_dict.values() for surname in names]
    name_combinations = []
    
    names_per_title = 102 // len(titles)  # Divide 102 equally among titles

    for title in titles:
        sampled_surnames = random.sample(all_surnames, names_per_title)
        for surname in sampled_surnames:
            name_combinations.append(f"{title} {surname}")

    return name_combinations

name_combinations = generate_name_combinations(surnames_dict)

# Print generated name combinations
for name in name_combinations:
    print(name)

In [ ]:
def questionsforhumans(name, question):
    response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo-0613",
        messages=[
            {
                "role": "user",
                "content": f""" 
                {name} is asked the following question: {question} 
                {name} answers:
                """
            },
        ],
        temperature=0,
    )
    return response

for name in name_combinations:
    name = name
    question = "What is the speed of sound in the air (in meters per second)?"
    response = questionsforhumans(name, question)
    answer = response['choices'][0]['message']['content']
    print(f"{name} answered: {answer}")

The findings of the code above clearly shows a problem where there is this problem of llms to act like real humans this is worth investigating more